# Track A challenge evaluator

Independent evaluator: SAM segmentation distance, public realism metrics, Moondream agreement,
diversity, efficiency, and Pareto ranking. It writes only to `track_a_evaluation_output`.

Every metric here is a continuous distance or agreement score, not a pass/fail gate — no metric
family excludes a scene or blocks another family from running. SAM, realism, and Moondream are
three independent axes; ranking (Pareto front / tie-break) compares contestants against each
other, it does not reject any single one.

In [1]:
import gc
import importlib
import json
import sys
from pathlib import Path

VENV_PYTHON = (Path.cwd() / '.venv' / 'bin' / 'python').resolve()
if Path(sys.executable).resolve() != VENV_PYTHON:
    raise RuntimeError('Select the Sim2Real (.venv) kernel and restart the notebook.')

import track_a_metrics
importlib.reload(track_a_metrics)
from image_metrics_utils import MoondreamAnalyzer, TransformersSamSegmenter, save_report
from track_a_metrics import *

OUTPUT = Path('track_a_evaluation_output')
OUTPUT.mkdir(exist_ok=True)
contestants = discover_contestants()
reports = {name: {'contestant': name} for name in contestants}
print(f'Found {len(contestants)} contestant(s): {list(contestants)}')


Found 1 contestant(s): ['provided_data_baseline']


## SAM segmentation distance

Runs class-agnostic SAM on real vs generated and simulated vs generated. Reports boundary F1,
symmetric region covering, matched IoU, ARI, and NMI as continuous distances — there is no
threshold and nothing here is excluded from later stages.

In [ ]:
sam = TransformersSamSegmenter('facebook/sam-vit-base')
for index, (name, folders) in enumerate(contestants.items(), 1):
    print(f'Contestant {index}/{len(contestants)}: SAM distance for {name}')
    reports[name]['sam'] = evaluate_sam(folders, OUTPUT / name / 'sam', sam)
    save_report(reports[name], OUTPUT / f'{name}.json')
del sam
gc.collect()
print('SAM distance complete.')

## 1. Raw public image metrics

All paired metrics from the ICST 2024 work are public. Generated vs real and simulated vs real are reported separately; FID, KID, and Inception Score are included as additional public distribution metrics.

In [3]:
for index, (name, folders) in enumerate(contestants.items(), 1):
    print(f'Contestant {index}/{len(contestants)}: public realism for all scenes in {name}')
    reports[name]['realism_raw'] = evaluate_public_realism(folders)
    save_report(reports[name], OUTPUT / f'{name}.json')
print('Public realism metrics complete.')


KeyError: 'sam'

## 2. Raw public Moondream failure rate

Queries categorical scene features and counts a scene as failing when at least one expected feature does not match. Replace this baseline adapter with the private industrial model during final evaluation.

In [ ]:
# Reload so this block always uses the current deterministic Moondream implementation.
import image_metrics_utils
importlib.reload(image_metrics_utils)
importlib.reload(track_a_metrics)
MoondreamAnalyzer = image_metrics_utils.MoondreamAnalyzer
evaluate_failure_rate = track_a_metrics.evaluate_failure_rate

# CPU, a pinned seed/revision, and greedy decoding make results reproducible.
moondream = MoondreamAnalyzer(revision='2025-06-21', device='cpu', seed=0)
for index, (name, folders) in enumerate(contestants.items(), 1):
    print(f'Contestant {index}/{len(contestants)}: failure evaluation for {name}')
    failure_report = evaluate_failure_rate(folders, moondream)
    reports[name]['failure_raw'] = failure_report
    print(f"  Failure rate: {failure_report['failure_rate']:.3f} "
          f"({failure_report['failures']}/{failure_report['executed']})")
    save_report(reports[name], OUTPUT / f'{name}.json')
del moondream
gc.collect()
print('Failure-rate evaluation complete.')


## 3. Final report and diversity

Assembles the sam / realism_raw / failure_raw sections into one final report per contestant.
Nothing is filtered or excluded here — this just adds feature diversity and (for scenes where
Moondream disagreed with the real reference) CLIP visual diversity on top of the raw sections.

In [ ]:
for index, (name, folders) in enumerate(contestants.items(), 1):
    print(f'Contestant {index}/{len(contestants)}: diversity for {name}')
    labels = load_labels(folders['labels'])
    reports[name]['final'] = build_final_report(reports[name], labels)
    failing_stems = [row['image'] for row in reports[name]['final']['failure']['images'] if row['failure']]
    reports[name]['final']['clip_visual_diversity'] = clip_visual_diversity(folders['generated'], failing_stems)
    save_report(reports[name], OUTPUT / f'{name}.json')
print('Diversity evaluation complete.')

## 4. Efficiency and ranking

Efficiency reads organizer-measured `generation_times.json`. Pareto ranking is produced only when every contestant has standardized timing.

In [ ]:
ranking_rows = []
for name, folders in contestants.items():
    final = reports[name]['final']
    executed = final['failure']['executed']
    reports[name]['efficiency'] = efficiency(folders['generated'].parent, executed)
    realism = final['realism']['generated_vs_real']['summary'].get('ssim')
    feature_div = final['feature_diversity']['mean_pairwise_hamming']
    timing = reports[name]['efficiency'].get('seconds_per_scene')
    ranking_rows.append({'contestant': name, 'failure': final['failure'].get('failure_rate'),
                         'realism': realism, 'diversity': feature_div, 'efficiency': timing})
    save_report(reports[name], OUTPUT / f'{name}.json')

rankable = [row for row in ranking_rows if all(row[key] is not None for key in ('failure', 'realism', 'diversity', 'efficiency'))]
fronts = pareto_front(rankable, {'failure': 'max', 'realism': 'max', 'diversity': 'max', 'efficiency': 'min'}) if rankable else []
tie_scores = tie_break_scores(rankable) if rankable else {}
leaderboard = {'rows': ranking_rows, 'pareto_fronts': fronts, 'tie_break_scores': tie_scores,
               'tie_break_weights': {'failure': 0.3, 'realism': 0.4, 'diversity': 0.3},
               'note': 'Tie-break values are min-max normalized across rankable contestants. '
                       'This ranks contestants relative to each other; it does not exclude any '
                       'contestant or scene from the metrics above.'}
save_report(leaderboard, OUTPUT / 'leaderboard.json')
leaderboard

## 5. Readable saved results

This cell reads saved reports only and can be run independently.

In [ ]:
for path in sorted(OUTPUT.glob('*.json')):
    if path.name == 'leaderboard.json':
        continue
    report = json.loads(path.read_text())
    print(f"\n{report['contestant']}")
    final = report.get('final')
    if final is None:
        raw = report.get('failure_raw', {})
        print(f"  Final report not assembled yet; raw Moondream scenes: {raw.get('executed', 0)}")
        continue
    failure = final['failure']
    failure_rate = failure.get('failure_rate')
    failure_text = f'{failure_rate:.3f}' if failure_rate is not None else 'N/A'
    print(f"  Moondream disagreement rate: {failure_text} ({failure['failures']}/{failure['executed']})")
    generated = final['realism']['generated_vs_real']['summary']
    simulated = final['realism']['simulated_vs_real']['summary']
    generated_ssim = generated.get('ssim')
    simulated_ssim = simulated.get('ssim')
    generated_text = f'{generated_ssim:.3f}' if generated_ssim is not None else 'N/A'
    simulated_text = f'{simulated_ssim:.3f}' if simulated_ssim is not None else 'N/A'
    print(f"  SSIM to real: generated {generated_text}, simulated {simulated_text}")
    sam = report.get('sam', {})
    sim_sam = sam.get('simulated_vs_generated', {}).get('summary', {}).get('partition', {})
    real_sam = sam.get('real_vs_generated', {}).get('summary', {}).get('partition', {})
    print(f"  SAM boundary F1: simulated_vs_generated {sim_sam.get('boundary_f1', float('nan')):.3f}, "
          f"real_vs_generated {real_sam.get('boundary_f1', float('nan')):.3f}")
    print(f"  Feature diversity: {final['feature_diversity']['mean_pairwise_hamming']:.3f}")